In [1]:
import pandas as pd

# Set pandas display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 100)        # Set the display width to 1000 characters
pd.set_option('display.max_colwidth', None) # Allow the full content of each column to be displayed

In [2]:
df_final_audit = pd.read_csv('./comprehensive_audit_final.csv')

In [6]:
import os, json, time, sys
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── 설정 ─────────────────────────────────────────────────
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY') or "sk-여기에직접입력"
client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

df_final_audit = pd.read_csv('./sfscore_df.csv', encoding='utf-8-sig')
all_vars = [col.replace('원본_', '') for col in df_final_audit.columns if col.startswith('원본_')]

np.random.seed(42)
df_sample = df_final_audit.sample(n=500).reset_index(drop=True)
print(f"샘플: {len(df_sample)}건 | 병렬 처리 시작")

SYSTEM_PROMPT = (
    "당신은 보험 언더라이팅 전문가입니다. "
    "주어진 의료 소견서에서 아래 변수들을 정확히 추출하여 JSON 형식으로만 반환하십시오. "
    "다른 텍스트는 출력하지 마십시오."
)

def make_user_prompt(sogyeon, var_list):
    return (
        f"다음 의료 소견서에서 아래 변수들을 추출하여 JSON으로 반환하십시오.\n"
        f"변수 목록: {var_list}\n\n"
        f"소견서:\n{sogyeon}\n\n"
        f'응답 형식: {{"변수명": "값", ...}}'
    )

def extract_one(idx_row):
    idx, row = idx_row
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": make_user_prompt(row['합성_소견서'], all_vars)}
            ],
            temperature=0,
            max_tokens=1024,
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        extracted = json.loads(raw)
        extracted['개인아이디'] = row['개인아이디']
        return idx, extracted, None
    except Exception as e:
        return idx, {'개인아이디': row['개인아이디'], '_error': str(e)}, str(e)

# ── 병렬 실행 (10개 동시) ────────────────────────────────
results = [None] * len(df_sample)
errors = 0
done = 0

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(extract_one, (i, row)): i
               for i, row in df_sample.iterrows()}
    for future in as_completed(futures):
        idx, result, err = future.result()
        results[idx] = result
        done += 1
        if err:
            errors += 1
        if done % 50 == 0:
            print(f"완료: {done}/500건 | 오류: {errors}건", flush=True)

print(f"\n추출 완료. 성공: {done-errors}건 / 오류: {errors}건")
df_gpt = pd.DataFrame(results)

# ── Sf 계산 ──────────────────────────────────────────────
CLINICAL_TOLERANCE = {
    '최종 수축기 혈압': 5.0, '최종 이완기 혈압': 5.0,
    '체질량지수': 0.5, '허리둘레': 1.0, '공복혈당': 5.0,
    '당화혈색소': 0.2, '총콜레스테롤': 5.0,
    'HDL-콜레스테롤': 3.0, '중성지방': 5.0,
    'LDL-콜레스테롤(직접검사)': 5.0,
}
clinical_cols = ['최종 수축기 혈압', '최종 이완기 혈압', '공복혈당', '체질량지수']
w_c, w_l = 0.68, 0.32
n_life = len(all_vars) - len(clinical_cols)

def parse_num(val):
    try:
        return float(str(val).replace(' ', '').replace(',', '.')
                     .split('m')[0].split('k')[0].split('%')[0])
    except:
        return None

def calc_sf_cross(orig_row, gpt_row):
    scores = []
    for var in all_vars:
        orig = orig_row.get(f'원본_{var}')
        recon = gpt_row.get(var)
        w = (w_c / len(clinical_cols)) if var in clinical_cols else (w_l / n_life)
        o, r = parse_num(orig), parse_num(recon)
        if o is not None and r is not None:
            tol = CLINICAL_TOLERANCE.get(var, 5.0)
            scores.append((1.0 if abs(o - r) <= tol else 0.0) * w)
        else:
            scores.append((1.0 if str(orig).strip() == str(recon).strip() else 0.0) * w)
    return sum(scores)

df_gpt_valid = df_gpt[~df_gpt.get('_error', pd.Series([None]*len(df_gpt))).notna()].copy() \
    if '_error' in df_gpt.columns else df_gpt.copy()
df_merged = df_sample.merge(df_gpt_valid, on='개인아이디', how='inner')
df_merged['Sf_cross'] = df_merged.apply(lambda row: calc_sf_cross(row, row), axis=1)

# ── 결과 출력 ────────────────────────────────────────────
print("\n" + "=" * 60)
print("Cross-model Sf 검증 결과 (Qwen 생성 → GPT-4o mini 추출)")
print("=" * 60)
print(f"유효 샘플 수          : {len(df_merged)}건")
print(f"Qwen Sf (동일 샘플)   : {df_merged['Sf_Score'].mean():.4f}")
print(f"GPT-4o mini Sf        : {df_merged['Sf_cross'].mean():.4f}")
print(f"Sf 차이               : {df_merged['Sf_Score'].mean() - df_merged['Sf_cross'].mean():.4f}")
print(f"GPT Sf ≥ 0.9 비율     : {(df_merged['Sf_cross'] >= 0.9).mean()*100:.1f}%")
print(f"GPT Sf = 1.0 비율     : {(df_merged['Sf_cross'] == 1.0).mean()*100:.1f}%")
print(f"Qwen Sf = 1.0 비율    : {(df_merged['Sf_Score'] == 1.0).mean()*100:.1f}%")

df_merged[['개인아이디', 'Sf_Score', 'Sf_cross']].to_csv(
    'cross_model_sf_result.csv', index=False, encoding='utf-8-sig'
)
print("\n결과 저장: cross_model_sf_result.csv")

샘플: 500건 | 병렬 처리 시작
완료: 50/500건 | 오류: 0건
완료: 100/500건 | 오류: 0건
완료: 150/500건 | 오류: 0건
완료: 200/500건 | 오류: 0건
완료: 250/500건 | 오류: 0건
완료: 300/500건 | 오류: 0건
완료: 350/500건 | 오류: 0건
완료: 400/500건 | 오류: 0건
완료: 450/500건 | 오류: 0건
완료: 500/500건 | 오류: 0건

추출 완료. 성공: 500건 / 오류: 0건

Cross-model Sf 검증 결과 (Qwen 생성 → GPT-4o mini 추출)
유효 샘플 수          : 500건
Qwen Sf (동일 샘플)   : 0.9964
GPT-4o mini Sf        : 0.8978
Sf 차이               : 0.0986
GPT Sf ≥ 0.9 비율     : 55.4%
GPT Sf = 1.0 비율     : 0.0%
Qwen Sf = 1.0 비율    : 0.0%

결과 저장: cross_model_sf_result.csv


In [7]:
# 소견서 샘플 1건 전체 출력
print(df_final_audit['합성_소견서'].iloc[0])

### 건강검진 소견서

#### 1. 신체계측 및 혈압
수축기 혈압 133 mmHg, 이완기 혈압 76 mmHg로 경계치입니다. 체질량지수는 26.9 kg/m²로 1단계 비만이며, 허리둘레 88.8 cm로 복부비만 기준을 초과하고 있습니다.

#### 2. 대사 지표
공복혈당 102 mg/dL, 당화혈색소 5.8%로 경계치입니다. 총콜레스테롤 230 mg/dL, HDL-콜레스테롤 43 mg/dL, LDL-콜레스테롤 146 mg/dL, 중성지방 274 mg/dL로 중성지방이 높습니다.

#### 3. 생활습관 및 건강행태
1년간 체중 증가가 관찰되었습니다. 음주는 월 2~4회, 한번에 7~9잔 정도입니다. 현재는 담배를 피우지 않고 있지만 과거에는 흡연하였었습니다. 유산소 운동을 실천하지 않고 있으며, 아침식사는 주 1~2회로 적습니다. 스트레스는 조금 느끼는 편입니다.

#### 4. 약물 복용 및 질환 유병 현황
혈압약, 당뇨약, 이상지질혈증약 모두 복용하지 않습니다. 당뇨병, 고혈압 유병 여부는 확인되지 않았지만, 고콜레스테롤혈증은 정상입니다.

#### 5. 종합 의견
전반적으로 일부 경계치 수치와 생활습관 개선이 필요합니다. 체중 감소와 유산소 운동 실천, 규칙적인 식사 습관을 유지하시고, 연 1회 정기검진을 권장합니다.

---

[CLINICAL_DATA_RECORD]
- 최종 수축기 혈압: 133.0 mmHg
- 최종 이완기 혈압: 76.0 mmHg
- 체질량지수: 26.91984854426037 kg/m²
- 허리둘레: 88.8 cm
- 공복혈당: 102.0 mg/dL
- 당화혈색소: 5.8%
- 총콜레스테롤: 230.0 mg/dL
- HDL-콜레스테롤: 43.0 mg/dL
- 중성지방: 274.0 mg/dL
- LDL-콜레스테롤(직접검사): 146.0 mg/dL
- 비만 유병여부: 1단계비만
- 1년간 체중 변화 여부: 체중 증가
- 1년간 음주빈도: 월 2~4회
- 한번에 마시는 음주량: 7~9잔
- 현재 일반담배 흡연 여부: 과거에는 피웠